<a href="https://colab.research.google.com/github/Akshatha7710/smart-tea-estate-management-system/blob/fertilizer-scheduling-and-optimisation-model/FerilizerSchedule.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount ('/content/drive')

Mounted at /content/drive


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt


Cleaning Data and Data Preprocessing

In [31]:
#fertilizer dataset
fert = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/DSGP/Fertilizer_History.csv', header=None)
fert = fert.dropna(how='all')

fert.columns = ["Division", "Field", "Extent", "Type", "Date", "Months", "Amount", "Extra"]
fert = fert.drop(columns=["Extra"])

# Preprocessing
fert['Date'] = pd.to_datetime(fert['Date'], errors='coerce')
fert['Division'] = fert['Division'].fillna(method='ffill')
fert['Amount'] = fert['Amount'].fillna(fert['Amount'].median())
fert['Fertilizer_per_area'] = fert['Amount'] / fert['Extent']

fert = fert.sort_values(['Division', 'Field', 'Date']).reset_index(drop=True)
fert['Days_since_last'] = fert.groupby(['Division','Field'])['Date'].diff().dt.days.fillna(0)
fert['Prev_Amount'] = fert.groupby(['Division','Field'])['Amount'].shift(1).fillna(0)

if 'Type' in fert.columns:
    fert = pd.get_dummies(fert, columns=['Type'], drop_first=True)

# Monthly aggregation
fert['Month'] = fert['Date'].dt.to_period('M')
fert_monthly = fert.groupby(['Division','Field','Month']).agg({
    'Amount':'sum',
    'Fertilizer_per_area':'mean',
    'Days_since_last':'mean',
    'Prev_Amount':'sum'
}).reset_index()

In [32]:
#Yield dataset
yield_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/DSGP/Block_Metadata.csv', header=None)
yield_df = yield_df.dropna(how='all').reset_index(drop=True)

yield_df = yield_df.iloc[4:].reset_index(drop=True)

yield_df.columns = [f'col_{i}' for i in range(len(yield_df.columns))]
yield_df = yield_df.rename(columns={'col_0':'Division', 'col_2':'Field', 'col_8':'Yield'})

yield_df['Yield'] = yield_df['Yield'].astype(str).str.replace(',', '', regex=False)
yield_df['Yield'] = pd.to_numeric(yield_df['Yield'], errors='coerce')
yield_df['Yield'] = yield_df['Yield'].fillna(yield_df['Yield'].median())

In [33]:
#Climate dataset
climate = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/DSGP/Climate_Data.csv')
climate = climate.dropna(how='all').reset_index(drop=True)

climate.columns = [f'col_{i}' for i in range(len(climate.columns))]

climate_data = climate.iloc[5:, :].reset_index(drop=True)
climate_data.columns = ['Month'] + [f'Year_{i}' for i in range(1, len(climate_data.columns))]

for col in climate_data.columns[1:]:
    climate_data[col] = pd.to_numeric(climate_data[col], errors='coerce')

climate_data = climate_data.fillna(climate_data.median(numeric_only=True))

Feature Engineering

In [34]:
fert['Prev_Amount'] = fert.groupby(['Division','Field'])['Amount'].shift(1).fillna(0)

fert['Month'] = fert['Date'].dt.to_period('M')

fert_monthly = fert.groupby(['Division','Field','Month']).agg({
    'Amount': 'sum',
    'Fertilizer_per_area': 'mean',
}).reset_index()

#Merge fertilizer features with yield data
yield_df = yield_df.rename(columns={'col_0':'Division', 'col_2':'Field', 'col_8':'Yield'})

data = yield_df.merge(fert_monthly, on=['Division','Field'], how='left')


print("\nFEATURE ENGINEERED DATA:")
print(data.head())


FEATURE ENGINEERED DATA:
  Division               col_1 Field col_3 col_4  col_5      col_6 col_7  \
0      LVO  Mr.R.W.M.Dilan R.B     1     4    VP  1,999  25-Sep-23    24   
1      NaN                 NaN     2     9    VP  1,999  07-Apr-24    17   
2      NaN                 NaN     3     5    VP  1,999  10-Feb-25     7   
3      NaN                 NaN    3B     5    VP  1,999  26-Mar-24    18   
4      NaN                 NaN     5     7    VP  1,999  19-Aug-24    13   

    Yield  col_9  ... col_14 col_15 col_16 col_17 col_18 col_19 col_20  \
0   943.0  2,036  ...  1,148  1,843  1,880   1489   1369    NaN    NaN   
1   879.0    843  ...  1,455  1,628  1,960   1193   1415    NaN    NaN   
2   452.0    967  ...  1,783  1,938  1,315   1363   1931    NaN    NaN   
3   919.0  1,120  ...  1,951  1,876  1,489   1290   1808    NaN    NaN   
4  1000.0    963  ...  1,862  2,307  2,726    721   1891    NaN    NaN   

     Month Amount  Fertilizer_per_area  
0  2023-09    0.0              